# Driver breaks and overnight rests

Long-haul routes are subject to driving-time regulations. Under the European Regulation (EC) No 561/2006, for example, a driver must take a **45 minute break after 4.5 hours of continuous driving**, and a **daily (overnight) rest of 11 hours once the daily driving limit of 9 hours is reached**. PyVRP's optimizer does not model these rules directly, but the `pyvrp.breaks` module can assign the required breaks and overnight rests *between a route's stops*, so that the resulting schedule complies with the regulations.

The rules are fully configurable through the `DriverRules` object (the defaults, `EU_RULES`, encode the common EU limits in minutes), so other regimes or custom company policies can be modelled just as easily. Rests may be taken anywhere en route, and routes may span multiple days.

In [ ]:
import numpy as np

from pyvrp import Model
from pyvrp.breaks import EU_RULES, plan_breaks, solve_with_breaks
from pyvrp.stop import MaxRuntime

## A long-haul instance

We build a small instance with clients spread out along a corridor. Distances are in kilometres and durations in minutes (roughly one minute per kilometre). The single vehicle has a shift spanning a full week, so its route may run over several days.

In [ ]:
coords = np.array([
    [0, 0],    # depot
    [120, 0],
    [260, 0],
    [400, 0],
    [560, 0],
    [720, 0],
])

m = Model()
locations = [m.add_location(x=int(x), y=int(y)) for x, y in coords]
depot = m.add_depot(locations[0])
for loc in locations[1:]:
    m.add_client(loc, service_duration=20)

# One vehicle, with a shift spanning a full week (in minutes).
m.add_vehicle_type(1, start_depot=depot, end_depot=depot, tw_late=7 * 24 * 60)

for i, frm in enumerate(locations):
    for j, to in enumerate(locations):
        if i != j:
            km = int(abs(coords[i, 0] - coords[j, 0]))
            m.add_edge(frm, to, distance=km, duration=km)

res = m.solve(stop=MaxRuntime(1), display=False)
data = m.data()
print(res.summary())

## Planning breaks for a solved route

`plan_breaks` walks each route's timeline and inserts the breaks and overnight rests required by the given rules. It returns a `CompliantSchedule` per route: an augmented timeline of stops interleaved with breaks and daily rests, plus summary statistics. Here we use the EU defaults.

In [ ]:
schedules = plan_breaks(res, data, EU_RULES)
for schedule in schedules:
    print(schedule)
    print(
        f"\ndriving={schedule.total_driving} min, "
        f"breaks={schedule.num_breaks}, "
        f"overnight rests={schedule.num_daily_rests}, "
        f"days={schedule.num_days}, feasible={schedule.is_feasible}\n"
    )

Each `BREAK` and `DAILY_REST` entry in the printed schedule sits *between* two stops (or part-way along a leg), at the exact point a driving limit is reached. The counters reset after each rest, and a new day begins after every overnight rest.

Because the optimizer solved the problem without knowing about breaks, inserting the required rest could in principle push a later stop past its time window. When that happens the schedule reports it through its `time_warp` and `is_feasible` attributes.

## Solving with breaks in one step

`solve_with_breaks` combines both steps: it inflates the travel durations so the optimizer reserves room for rest, solves, plans the exact rests, and re-solves with more reserved room if inserting rest would break a time window. It returns the usual `Result` together with a compliant schedule per route.

In [ ]:
result, schedules = solve_with_breaks(m, MaxRuntime(1), EU_RULES)
print("all schedules feasible:", all(s.is_feasible for s in schedules))
for schedule in schedules:
    print(schedule)